# MVP: Sistema de Suporte à Decisão para a Segurança nas Rodovias Federais

**Aluno:** Pedro Augusto de Menezes Filho (180114425)
**Disciplina:** Sistemas de Suporte à Decisão, Departamento de Engenharia de Produção, UnB
**Professor:** André Luiz Marques Serrano
**Plataforma:** Google Colab (Python) com os dados persistidos no Google Drive

---

## 1. Objetivo

A Polícia Rodoviária Federal (PRF) tem efetivo, viaturas e equipes de socorro limitados para
cobrir mais de 70 mil km de rodovias federais. Dois tipos de decisão se repetem todos os dias:

1. **Planejamento:** em quais trechos, dias e horários concentrar a fiscalização.
2. **Atendimento:** quando um acidente é comunicado, se ele deve receber socorro prioritário
   (por exemplo, uma equipe de resgate avançado), antes de se saber quantas vítimas graves houve.

Em Engenharia de Produção, é um problema clássico de **alocação de recursos escassos**: colocar
a capacidade onde o risco é maior.

**Hipótese:** os acidentes graves (com morto ou ferido grave) não se distribuem ao acaso.
Eles se concentram em poucos trechos, em certos horários e em certos tipos de acidente e de
vítima. Por isso, (a) uma pequena parte dos trechos cobre uma grande parte dos acidentes graves e
(b) um modelo de *machine learning* consegue indicar, com as informações do chamado, quais
acidentes têm mais chance de serem graves, melhor do que uma regra simples.

### Perguntas de negócio e critérios definidos antes da análise

| Nº | Pergunta | Critério (C) |
|----|----------|--------------|
| P1 | **Onde** se concentram os acidentes graves? | C1: 10% dos trechos de 10 km com acidentes concentram ≥ 40% dos acidentes graves |
| P2 | **Quando** os acidentes são mais graves? | C2: a taxa de gravidade à noite é ≥ 1,2 vez a taxa de dia |
| P3 | **Como** acontecem os acidentes que mais matam? | C3: 3 tipos de acidente respondem por ≥ 50% das mortes |
| P4 | **Quem** são as vítimas mais expostas? | C4: a letalidade de pedestres e de motociclistas é ≥ 2 vezes a média dos envolvidos |
| P5 | Um modelo de ML prevê a gravidade melhor que uma regra simples? | C5: sinalizando 30% dos acidentes, o melhor modelo captura ≥ 5 pontos percentuais a mais de acidentes graves que a regra "taxa histórica do tipo de acidente" |

A hipótese é confirmada se os cinco critérios forem atendidos.

## 2. Configuração do ambiente

Bibliotecas: `pandas`, `numpy`, `matplotlib` e `scikit-learn` (já vêm instaladas no Colab).

Os dados ficam no **Google Drive**, em camadas:
- **bronze:** arquivos brutos, exatamente como foram baixados;
- **prata:** dados limpos e padronizados;
- **ouro:** esquema estrela pronto para análise;
- **catalogo** e **evidencias:** catálogo de dados, relatórios e figuras.

In [ ]:
import os
import datetime
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dateutil.easter import easter

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

AZUL = '#2a78d6'
LARANJA = '#e8673a'
CINZA = '#7a7a7a'
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 10})

In [ ]:
# No Colab, monta o Google Drive. Fora do Colab, usa uma pasta local com a mesma estrutura.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/MVP_PRF_Acidentes'
except ImportError:
    BASE = os.environ.get('MVP_BASE', 'MVP_PRF_Acidentes')

BRONZE = os.path.join(BASE, '1_bronze')
PRATA = os.path.join(BASE, '2_prata')
OURO = os.path.join(BASE, '3_ouro')
CATALOGO = os.path.join(BASE, 'catalogo')
EVIDENCIAS = os.path.join(BASE, 'evidencias')
for pasta in [BRONZE, PRATA, OURO, CATALOGO, EVIDENCIAS]:
    os.makedirs(pasta, exist_ok=True)
print('Pasta do projeto:', BASE)

## 3. Busca e coleta dos dados (camada bronze)

**Fonte:** Dados Abertos da PRF (https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf),
também listados no Portal Brasileiro de Dados Abertos. São usados dois conjuntos, para 2024 e 2025:

| Conjunto | Grão (o que é cada linha) |
|---|---|
| Acidentes agrupados por ocorrência (`datatran`) | um acidente |
| Acidentes agrupados por pessoa, todas as causas e tipos (`acidentes`) | uma pessoa envolvida, repetida para cada causa e tipo de acidente registrado |

**Licença:** dados abertos do governo federal (Decreto nº 8.777/2016), de uso, reutilização e
redistribuição livres, com citação da fonte.

**Como os arquivos são baixados:** a página da PRF publica cada arquivo como um `.zip` no Google
Drive. O código abaixo baixa esses arquivos pelos identificadores publicados na página. Se o
download falhar (o Google Drive às vezes bloqueia downloads automáticos), o código usa uma cópia
pública dos mesmos arquivos em formato Parquet, mantida no GitHub pelo projeto *tidyprf-dados*,
que converte os arquivos oficiais sem alterar os registros. A origem usada fica no registro da coleta.

In [ ]:
ANOS = [2024, 2025]

# Identificadores dos arquivos no Google Drive, copiados da página de dados abertos da PRF
ID_DRIVE = {
    ('datatran', 2024): '14lB0vqMFkaZj8HZ44b0njYgxs9nAN8KO',
    ('datatran', 2025): '1-G3MdmHBt6CprDwcW99xxC4BZ2DU5ryR',
    ('acidentes', 2024): '14qBOhrE1gioVtuXgxkCJ9kCA8YtUGXKA',
    ('acidentes', 2025): '1-PJGRbfSe7PVjU37A3wTCls_NRXyVGRD',
}
URL_DRIVE = 'https://drive.usercontent.google.com/download?id={}&export=download&confirm=t'
URL_COPIA = 'https://github.com/bonijoao/tidyprf-dados/releases/download/dados-v1/{}_{}.parquet'

registro = []
for (conjunto, ano), id_arquivo in ID_DRIVE.items():
    arquivo_zip = os.path.join(BRONZE, f'{conjunto}_{ano}.zip')
    arquivo_parquet = os.path.join(BRONZE, f'{conjunto}_{ano}.parquet')
    if os.path.exists(arquivo_zip) or os.path.exists(arquivo_parquet):
        origem = 'arquivo já existente na pasta bronze'
    else:
        try:
            urllib.request.urlretrieve(URL_DRIVE.format(id_arquivo), arquivo_zip)
            pd.read_csv(arquivo_zip, sep=';', encoding='latin-1', nrows=5)  # testa se é um zip com CSV
            origem = 'PRF (Google Drive oficial)'
        except Exception as erro:
            print(f'{conjunto} {ano}: download oficial falhou ({erro}). Usando a cópia do GitHub.')
            if os.path.exists(arquivo_zip):
                os.remove(arquivo_zip)
            urllib.request.urlretrieve(URL_COPIA.format(conjunto, ano), arquivo_parquet)
            origem = 'cópia tidyprf-dados (GitHub)'
    caminho = arquivo_zip if os.path.exists(arquivo_zip) else arquivo_parquet
    registro.append({'conjunto': conjunto, 'ano': ano, 'arquivo': os.path.basename(caminho),
                     'origem': origem, 'data_coleta': datetime.date.today().isoformat(),
                     'tamanho_mb': round(os.path.getsize(caminho) / 1024 / 1024, 2)})

In [ ]:
# Leitura dos arquivos brutos. Os CSVs da PRF usam ponto e vírgula, codificação latin-1 e
# vírgula como separador decimal; por isso tudo é lido primeiro como veio.
brutos = {'datatran': [], 'acidentes': []}
for item in registro:
    caminho = os.path.join(BRONZE, item['arquivo'])
    if caminho.endswith('.zip'):
        tabela = pd.read_csv(caminho, sep=';', encoding='latin-1', low_memory=False)
    else:
        tabela = pd.read_parquet(caminho)
    item['linhas'] = len(tabela)
    item['colunas'] = tabela.shape[1]
    brutos[item['conjunto']].append(tabela)

ocorrencias_bruto = pd.concat(brutos['datatran'], ignore_index=True)
pessoas_bruto = pd.concat(brutos['acidentes'], ignore_index=True)

registro_coleta = pd.DataFrame(registro)
registro_coleta.to_csv(os.path.join(BRONZE, 'registro_coleta.csv'), index=False)
print('Ocorrências brutas:', ocorrencias_bruto.shape, '| Pessoas brutas:', pessoas_bruto.shape)
registro_coleta

## 4. Qualidade dos dados brutos

Antes de qualquer tratamento, cada atributo usado é verificado nas seis dimensões de qualidade:
completude, unicidade, consistência, conformidade, acurácia e atualidade. Os problemas
encontrados aqui viram as regras de transformação da Seção 5.

In [ ]:
# Completude: valores nulos ou "Ignorado" por atributo
colunas_ocorrencia = ['id', 'data_inversa', 'horario', 'uf', 'br', 'km', 'tipo_acidente', 'causa_acidente',
                      'classificacao_acidente', 'fase_dia', 'condicao_metereologica', 'tipo_pista',
                      'tracado_via', 'uso_solo', 'pessoas', 'mortos', 'feridos_graves', 'veiculos',
                      'latitude', 'longitude']
colunas_pessoa = ['id', 'pesid', 'tipo_veiculo', 'tipo_envolvido', 'estado_fisico', 'idade', 'sexo',
                  'ano_fabricacao_veiculo']

completude = []
for nome, tabela, colunas in [('ocorrência', ocorrencias_bruto, colunas_ocorrencia),
                              ('pessoa', pessoas_bruto, colunas_pessoa)]:
    for coluna in colunas:
        texto = tabela[coluna].astype(str).str.strip()
        vazios = tabela[coluna].isna().sum() + texto.isin(['Ignorado', 'Não Informado', '(null)', 'NA', '']).sum()
        completude.append([nome, coluna, int(vazios), round(vazios / len(tabela) * 100, 2)])
completude = pd.DataFrame(completude, columns=['tabela', 'atributo', 'vazios_ou_ignorados', '%'])
completude[completude['vazios_ou_ignorados'] > 0]

In [ ]:
qualidade = []

# Unicidade
qualidade.append(['unicidade', 'ocorrência: id', f"{ocorrencias_bruto['id'].duplicated().sum()} ids repetidos"])
qualidade.append(['conformidade', 'ocorrência: br', f"{(pd.to_numeric(ocorrencias_bruto['br'], errors='coerce') == 0).sum()} acidentes com BR igual a 0 (rodovia inexistente)"])
sem_pesid = pessoas_bruto[pessoas_bruto['pesid'].isna()]
qualidade.append(['consistência', 'pessoa: pesid vazio',
                  f"{len(sem_pesid):,} linhas sem pessoa; tipos de veículo mais comuns: " +
                  ', '.join(sem_pesid['tipo_veiculo'].value_counts().index[:3])])
qualidade.append(['unicidade', 'pessoa: id + pesid',
                  f"{len(pessoas_bruto):,} linhas para {pessoas_bruto[['id', 'pesid']].dropna().drop_duplicates().shape[0]:,} pessoas "
                  f"(a mesma pessoa se repete para cada causa e tipo de acidente)"])

# Consistência entre as duas tabelas
ids_ocorrencia = set(ocorrencias_bruto['id'])
ids_pessoa = set(pessoas_bruto['id'])
qualidade.append(['consistência', 'id nas duas tabelas',
                  f"{len(ids_pessoa - ids_ocorrencia)} acidentes só na tabela de pessoas; "
                  f"{len(ids_ocorrencia - ids_pessoa)} só na de ocorrências"])

# Conformidade e acurácia
idade = pd.to_numeric(pessoas_bruto['idade'].astype(str).str.replace(',', '.'), errors='coerce')
ano_veiculo = pd.to_numeric(pessoas_bruto['ano_fabricacao_veiculo'].astype(str).str.replace(',', '.'), errors='coerce')
latitude = pd.to_numeric(ocorrencias_bruto['latitude'].astype(str).str.replace(',', '.'), errors='coerce')
longitude = pd.to_numeric(ocorrencias_bruto['longitude'].astype(str).str.replace(',', '.'), errors='coerce')
fora_brasil = ((latitude < -34) | (latitude > 6) | (longitude < -74) | (longitude > -34)).sum()
qualidade.append(['acurácia', 'pessoa: idade', f"{(idade > 100).sum()} idades acima de 100 anos (máximo {idade.max():.0f})"])
qualidade.append(['acurácia', 'pessoa: ano_fabricacao_veiculo', f"{(ano_veiculo < 1950).sum()} anos antes de 1950 (mínimo {ano_veiculo.min():.0f})"])
ano_bruto = pd.to_datetime(ocorrencias_bruto['data_inversa'].astype(str), errors='coerce').dt.year
fora_por_ano = ', '.join(f'{a}: {n:,}' for a, n in (((latitude < -34) | (latitude > 6) | (longitude < -74) | (longitude > -34))
                                                    .groupby(ano_bruto).sum().items()))
qualidade.append(['acurácia', 'ocorrência: latitude/longitude', f"{fora_brasil:,} pontos fora do território brasileiro ({fora_por_ano})"])
qualidade.append(['conformidade', 'ocorrência: condicao_metereologica',
                  'categorias sobrepostas: ' + ', '.join(sorted(ocorrencias_bruto['condicao_metereologica'].dropna().unique()))])
qualidade.append(['conformidade', 'ocorrência: classificacao_acidente',
                  ', '.join(f'{k}: {v}' for k, v in ocorrencias_bruto['classificacao_acidente'].value_counts(dropna=False).items())])

# Atualidade
datas = pd.to_datetime(ocorrencias_bruto['data_inversa'].astype(str), errors='coerce')
qualidade.append(['atualidade', 'ocorrência: data_inversa', f"de {datas.min().date()} a {datas.max().date()}"])

qualidade = pd.DataFrame(qualidade, columns=['dimensão', 'atributo', 'resultado'])
qualidade.to_csv(os.path.join(CATALOGO, 'relatorio_qualidade.csv'), index=False)
completude.to_csv(os.path.join(CATALOGO, 'relatorio_completude.csv'), index=False)
qualidade

## 5. Transformação (camada prata): regras de negócio

| Regra | O que faz | Por quê |
|---|---|---|
| R1 | Converte km, latitude, longitude e contagens para número (vírgula → ponto); coordenada fora do Brasil e BR igual a 0 viram nulo | O CSV usa vírgula decimal e há valores impossíveis |
| R2 | Converte data e horário; cria hora e faixa horária | Análise por horário |
| R3 | "Ignorado", "Não Informado" e "(null)" viram nulo | Não são categorias reais |
| R4 | Agrupa condições do tempo: Sol → Céu Claro; Garoa/Chuvisco → Chuva | Categorias sobrepostas |
| R5 | Tabela de pessoas: uma linha por pessoa em cada acidente (remove repetições por causa e tipo) e retira as linhas sem pessoa (reboques e semirreboques) | Define o grão |
| R6 | LGPD: idade vira faixa etária; saem pesid, naturalidade, nacionalidade, marca e id do veículo | Minimização de dados pessoais |
| R7 | Idade acima de 100 anos e ano do veículo antes de 1950 ou depois do ano do acidente viram nulo | Valores implausíveis |
| R8 | Conciliação: mortos, feridos graves e pessoas da ocorrência são comparados com a tabela de pessoas; o total de pessoas passa a ser o recontado | Garante que as duas tabelas contam a mesma coisa |
| R9 | **Acidente grave** = pelo menos 1 morto ou 1 ferido grave | Variável de decisão |
| R10 | **Trecho** = BR + UF + segmento de 10 km (ex.: BR-116/SP km 220–229) | Unidade de planejamento |
| R11 | Região do IBGE, fim de semana e feriados nacionais | Enriquecimento |
| R12 | Veículos e envolvidos por acidente: tem moto, tem pedestre, tem veículo pesado | Une a tabela de pessoas à de ocorrências |

In [ ]:
ocorrencias = ocorrencias_bruto.copy()

# R1: números
for coluna in ['km', 'latitude', 'longitude', 'br', 'pessoas', 'mortos', 'feridos_leves', 'feridos_graves',
               'ilesos', 'ignorados', 'feridos', 'veiculos']:
    ocorrencias[coluna] = pd.to_numeric(ocorrencias[coluna].astype(str).str.replace(',', '.'), errors='coerce')
ocorrencias['id'] = pd.to_numeric(ocorrencias['id'].astype(str).str.replace(',', '.'), errors='coerce').astype('Int64')
fora = ((ocorrencias['latitude'] < -34) | (ocorrencias['latitude'] > 6) |
        (ocorrencias['longitude'] < -74) | (ocorrencias['longitude'] > -34))
ocorrencias.loc[fora, ['latitude', 'longitude']] = np.nan
ocorrencias.loc[ocorrencias['br'] == 0, 'br'] = np.nan
print(f'R1: {fora.sum()} coordenadas fora do Brasil e {ocorrencias["br"].isna().sum()} BRs inválidas viraram nulo')

# R2: data e horário
ocorrencias['data'] = pd.to_datetime(ocorrencias['data_inversa'].astype(str), errors='coerce')
ocorrencias['hora'] = pd.to_numeric(ocorrencias['horario'].astype(str).str[:2], errors='coerce')
ocorrencias['faixa_horaria'] = pd.cut(ocorrencias['hora'], bins=[-1, 5, 11, 17, 23],
                                      labels=['madrugada (0-5h)', 'manhã (6-11h)', 'tarde (12-17h)', 'noite (18-23h)'])

# R3: valores que significam "não informado"
colunas_texto = ['uf', 'municipio', 'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 'fase_dia',
                 'sentido_via', 'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo']
for coluna in colunas_texto:
    ocorrencias[coluna] = ocorrencias[coluna].astype('string').str.strip()
    ocorrencias[coluna] = ocorrencias[coluna].replace(['Ignorado', 'Ignorada', 'Não Informado', '(null)', 'NA', ''], pd.NA)
ocorrencias['uf'] = ocorrencias['uf'].str.upper()

# R4: condições do tempo
ocorrencias['condicao_tempo'] = ocorrencias['condicao_metereologica'].replace({'Sol': 'Céu Claro', 'Garoa/Chuvisco': 'Chuva'})

print(ocorrencias.shape)
ocorrencias['condicao_tempo'].value_counts(dropna=False)

In [ ]:
pessoas = pessoas_bruto.copy()
pessoas['id'] = pd.to_numeric(pessoas['id'].astype(str).str.replace(',', '.'), errors='coerce').astype('Int64')

# R5: uma linha por pessoa em cada acidente
linhas_antes = len(pessoas)
pessoas = pessoas.drop_duplicates(['id', 'pesid'])
veiculos_sem_pessoa = pessoas[pessoas['pesid'].isna()]   # reboques e semirreboques: guardados para a R12
pessoas = pessoas[pessoas['pesid'].notna()]
print(f'R5: {linhas_antes:,} linhas -> {len(pessoas):,} pessoas (+ {len(veiculos_sem_pessoa):,} linhas de reboque sem pessoa)')

# R7: valores implausíveis
pessoas['idade'] = pd.to_numeric(pessoas['idade'].astype(str).str.replace(',', '.'), errors='coerce')
pessoas['ano_fabricacao_veiculo'] = pd.to_numeric(pessoas['ano_fabricacao_veiculo'].astype(str).str.replace(',', '.'), errors='coerce')
ano_acidente = pd.to_datetime(pessoas['data_inversa'].astype(str), errors='coerce').dt.year
idade_ruim = pessoas['idade'] > 100
veiculo_ruim = (pessoas['ano_fabricacao_veiculo'] < 1950) | (pessoas['ano_fabricacao_veiculo'] > ano_acidente)
pessoas.loc[idade_ruim, 'idade'] = np.nan
pessoas.loc[veiculo_ruim, 'ano_fabricacao_veiculo'] = np.nan
print(f'R7: {idade_ruim.sum()} idades e {veiculo_ruim.sum()} anos de veículo anulados')

# R6: LGPD - idade vira faixa e saem os identificadores e dados pessoais não usados
pessoas['faixa_etaria'] = pd.cut(pessoas['idade'], bins=[-1, 17, 29, 44, 59, 100],
                                 labels=['0-17', '18-29', '30-44', '45-59', '60+'])
for coluna in ['mortos', 'feridos_graves', 'feridos_leves', 'ilesos']:
    pessoas[coluna] = pd.to_numeric(pessoas[coluna], errors='coerce').fillna(0).astype(int)
for coluna in ['tipo_veiculo', 'tipo_envolvido', 'estado_fisico', 'sexo']:
    pessoas[coluna] = pessoas[coluna].astype('string').str.strip()
    pessoas[coluna] = pessoas[coluna].replace(['Ignorado', 'Não Informado', '(null)', 'NA', ''], pd.NA)
pessoas = pessoas[['id', 'tipo_veiculo', 'ano_fabricacao_veiculo', 'tipo_envolvido', 'estado_fisico',
                   'faixa_etaria', 'sexo', 'mortos', 'feridos_graves', 'feridos_leves', 'ilesos']]
print('Colunas mantidas na tabela de pessoas:', list(pessoas.columns))

In [ ]:
# R8: conciliação entre as duas tabelas
soma_pessoas = pessoas.groupby('id').agg(mortos_p=('mortos', 'sum'), feridos_graves_p=('feridos_graves', 'sum'),
                                         pessoas_p=('id', 'size'))
conferencia = ocorrencias.set_index('id')[['mortos', 'feridos_graves', 'pessoas']].join(soma_pessoas, how='left')
conciliacao = pd.DataFrame({
    'medida': ['mortos', 'feridos graves', 'pessoas'],
    'total_ocorrencias': [conferencia['mortos'].sum(), conferencia['feridos_graves'].sum(), conferencia['pessoas'].sum()],
    'total_pessoas': [conferencia['mortos_p'].sum(), conferencia['feridos_graves_p'].sum(), conferencia['pessoas_p'].sum()],
    'acidentes_divergentes': [(conferencia['mortos'] != conferencia['mortos_p']).sum(),
                              (conferencia['feridos_graves'] != conferencia['feridos_graves_p']).sum(),
                              (conferencia['pessoas'] != conferencia['pessoas_p']).sum()],
})
diferenca = conferencia['pessoas'] - conferencia['pessoas_p']
com_reboque = conferencia.index.isin(veiculos_sem_pessoa['id'])
print('Diferença de pessoas nos acidentes divergentes:', diferenca[diferenca != 0].value_counts().to_dict())
print('Acidentes divergentes que têm reboque sem pessoa:', (com_reboque & (diferenca != 0)).sum(), 'de', (diferenca != 0).sum())
# O total de pessoas da ocorrência conta 1 a mais quando há reboque; fica valendo o recontado
ocorrencias['pessoas'] = ocorrencias['id'].map(conferencia['pessoas_p']).fillna(0).astype(int)
conciliacao.to_csv(os.path.join(CATALOGO, 'conciliacao.csv'), index=False)
conciliacao

In [ ]:
# R9: acidente grave
ocorrencias['grave'] = ((ocorrencias['mortos'] > 0) | (ocorrencias['feridos_graves'] > 0)).astype(int)

# R10: trecho de 10 km
ocorrencias['km_inicial'] = (ocorrencias['km'] // 10 * 10)
tem_trecho = ocorrencias['br'].notna() & ocorrencias['km_inicial'].notna() & ocorrencias['uf'].notna()
ocorrencias['id_trecho'] = pd.NA
ocorrencias.loc[tem_trecho, 'id_trecho'] = ('BR-' + ocorrencias.loc[tem_trecho, 'br'].astype(int).astype(str) + '/' +
                                            ocorrencias.loc[tem_trecho, 'uf'] + ' km ' +
                                            ocorrencias.loc[tem_trecho, 'km_inicial'].astype(int).astype(str))

# R11: região do IBGE
regioes = {'Norte': ['AC', 'AM', 'AP', 'PA', 'RO', 'RR', 'TO'],
           'Nordeste': ['AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE'],
           'Centro-Oeste': ['DF', 'GO', 'MS', 'MT'],
           'Sudeste': ['ES', 'MG', 'RJ', 'SP'],
           'Sul': ['PR', 'RS', 'SC']}
for regiao, ufs in regioes.items():
    ocorrencias.loc[ocorrencias['uf'].isin(ufs), 'regiao'] = regiao

# R12: o que estava envolvido em cada acidente (vem da tabela de pessoas)
motos = ['Motocicleta', 'Motoneta', 'Ciclomotor']
pesados = ['Caminhão', 'Caminhão-trator', 'Ônibus', 'Micro-ônibus', 'Semireboque', 'Reboque']
veiculos_e_pessoas = pd.concat([pessoas[['id', 'tipo_veiculo', 'tipo_envolvido']],
                                veiculos_sem_pessoa[['id', 'tipo_veiculo', 'tipo_envolvido']]])
envolvidos = pd.DataFrame({
    'tem_moto': veiculos_e_pessoas['tipo_veiculo'].isin(motos).groupby(veiculos_e_pessoas['id']).max(),
    'tem_pedestre': (veiculos_e_pessoas['tipo_envolvido'] == 'Pedestre').groupby(veiculos_e_pessoas['id']).max(),
    'tem_pesado': veiculos_e_pessoas['tipo_veiculo'].isin(pesados).groupby(veiculos_e_pessoas['id']).max(),
}).astype(int)
ocorrencias = ocorrencias.merge(envolvidos, left_on='id', right_index=True, how='left')
for coluna in ['tem_moto', 'tem_pedestre', 'tem_pesado']:
    ocorrencias[coluna] = ocorrencias[coluna].fillna(0).astype(int)

ocorrencias.to_csv(os.path.join(PRATA, 'ocorrencias_prata.csv'), index=False)
pessoas.to_csv(os.path.join(PRATA, 'pessoas_prata.csv'), index=False)
print('Acidentes graves:', ocorrencias['grave'].sum(), f"({ocorrencias['grave'].mean() * 100:.1f}%)")
print('Acidentes sem trecho definido:', (~tem_trecho).sum())

## 6. Modelagem (camada ouro): esquema estrela

```
                dim_tempo
                    |
  dim_trecho --- fato_acidente --- fato_envolvido
```

- **fato_acidente** (grão: um acidente): medidas (pessoas, mortos, feridos, veículos, grave) e
  atributos do acidente (tipo, causa, fase do dia, condição do tempo e da via).
- **fato_envolvido** (grão: uma pessoa envolvida em um acidente): tipo de veículo, tipo de
  envolvido, estado físico, faixa etária e sexo.
- **dim_tempo** (grão: um dia): dia da semana, mês, fim de semana e feriados.
- **dim_trecho** (grão: 10 km de uma BR em uma UF): BR, UF, região e km inicial e final.

In [ ]:
# R11: feriados nacionais (datas fixas e móveis, calculadas a partir da Páscoa)
feriados = {}
for ano in ANOS:
    pascoa = pd.Timestamp(easter(ano))
    feriados[pd.Timestamp(f'{ano}-01-01')] = 'Confraternização Universal'
    feriados[pascoa - pd.Timedelta(days=48)] = 'Carnaval (segunda)'
    feriados[pascoa - pd.Timedelta(days=47)] = 'Carnaval (terça)'
    feriados[pascoa - pd.Timedelta(days=2)] = 'Sexta-feira Santa'
    feriados[pd.Timestamp(f'{ano}-04-21')] = 'Tiradentes'
    feriados[pd.Timestamp(f'{ano}-05-01')] = 'Dia do Trabalho'
    feriados[pascoa + pd.Timedelta(days=60)] = 'Corpus Christi'
    feriados[pd.Timestamp(f'{ano}-09-07')] = 'Independência'
    feriados[pd.Timestamp(f'{ano}-10-12')] = 'Nossa Senhora Aparecida'
    feriados[pd.Timestamp(f'{ano}-11-02')] = 'Finados'
    feriados[pd.Timestamp(f'{ano}-11-15')] = 'Proclamação da República'
    feriados[pd.Timestamp(f'{ano}-11-20')] = 'Consciência Negra'
    feriados[pd.Timestamp(f'{ano}-12-25')] = 'Natal'

nomes_dias = ['segunda', 'terça', 'quarta', 'quinta', 'sexta', 'sábado', 'domingo']
dim_tempo = pd.DataFrame({'data': pd.date_range(f'{ANOS[0]}-01-01', f'{ANOS[-1]}-12-31')})
dim_tempo['ano'] = dim_tempo['data'].dt.year
dim_tempo['mes'] = dim_tempo['data'].dt.month
dim_tempo['dia_semana'] = dim_tempo['data'].dt.dayofweek.map(dict(enumerate(nomes_dias)))
dim_tempo['fim_de_semana'] = dim_tempo['data'].dt.dayofweek >= 5
dim_tempo['feriado'] = dim_tempo['data'].isin(feriados.keys())
dim_tempo['nome_feriado'] = dim_tempo['data'].map(feriados)

# dim_trecho
dim_trecho = (ocorrencias.dropna(subset=['id_trecho'])
              .groupby('id_trecho', as_index=False)
              .agg(br=('br', 'first'), uf=('uf', 'first'), regiao=('regiao', 'first'), km_inicial=('km_inicial', 'first')))
dim_trecho['br'] = dim_trecho['br'].astype(int)
dim_trecho['km_final'] = dim_trecho['km_inicial'] + 9.9

# fato_acidente
fato_acidente = ocorrencias[['id', 'data', 'hora', 'faixa_horaria', 'id_trecho', 'km', 'municipio', 'latitude', 'longitude',
                             'tipo_acidente', 'causa_acidente', 'fase_dia', 'condicao_tempo', 'tipo_pista',
                             'tracado_via', 'uso_solo', 'pessoas', 'mortos', 'feridos_graves', 'feridos_leves',
                             'ilesos', 'veiculos', 'tem_moto', 'tem_pedestre', 'tem_pesado', 'grave']].copy()
fato_acidente = fato_acidente.rename(columns={'id': 'id_acidente'})

# fato_envolvido
fato_envolvido = pessoas.rename(columns={'id': 'id_acidente'})
fato_envolvido['obito'] = (fato_envolvido['mortos'] > 0).astype(int)
fato_envolvido = fato_envolvido.drop(columns=['mortos', 'feridos_graves', 'feridos_leves', 'ilesos'])

print('fato_acidente', fato_acidente.shape, '| fato_envolvido', fato_envolvido.shape,
      '| dim_tempo', dim_tempo.shape, '| dim_trecho', dim_trecho.shape)

### Catálogo de dados

Para cada atributo do modelo: tabela, nome, tipo, descrição, domínio (faixa ou categorias),
se admite nulo e linhagem (arquivo de origem, data de extração e regra aplicada).

In [ ]:
hoje = datetime.date.today().isoformat()
ORIGEM_OC = f'PRF datatran 2024-2025, extraído em {hoje}'
ORIGEM_PE = f'PRF acidentes por pessoa 2024-2025, extraído em {hoje}'

catalogo = pd.DataFrame([
    # tabela, atributo, tipo, descrição, mínimo, máximo, categorias, admite nulo, linhagem
    ['fato_acidente', 'id_acidente', 'numérico', 'Identificador do acidente na PRF', None, None, None, 'não', ORIGEM_OC + '; coluna id'],
    ['fato_acidente', 'data', 'data', 'Data do acidente', '2024-01-01', '2025-12-31', None, 'não', ORIGEM_OC + '; data_inversa; R2'],
    ['fato_acidente', 'hora', 'numérico', 'Hora do acidente', 0, 23, None, 'não', ORIGEM_OC + '; horario; R2'],
    ['fato_acidente', 'faixa_horaria', 'categórico', 'Faixa do dia', None, None, 'madrugada (0-5h), manhã (6-11h), tarde (12-17h), noite (18-23h)', 'não', 'Derivado de hora; R2'],
    ['fato_acidente', 'id_trecho', 'texto', 'Trecho de 10 km (chave para dim_trecho)', None, None, None, 'sim (sem km ou BR)', 'br + uf + km; R10'],
    ['fato_acidente', 'km', 'numérico', 'Quilômetro da rodovia', 0, 1500, None, 'sim', ORIGEM_OC + '; km; R1'],
    ['fato_acidente', 'latitude', 'numérico', 'Latitude do acidente', -34, 6, None, 'sim (fora do Brasil vira nulo)', ORIGEM_OC + '; latitude; R1'],
    ['fato_acidente', 'longitude', 'numérico', 'Longitude do acidente', -74, -34, None, 'sim', ORIGEM_OC + '; longitude; R1'],
    ['fato_acidente', 'tipo_acidente', 'categórico', 'Tipo do acidente (ex.: colisão frontal)', None, None, None, 'sim', ORIGEM_OC + '; tipo_acidente; R3'],
    ['fato_acidente', 'causa_acidente', 'categórico', 'Causa apontada pela PRF', None, None, None, 'sim', ORIGEM_OC + '; causa_acidente; R3'],
    ['fato_acidente', 'fase_dia', 'categórico', 'Luminosidade', None, None, 'Pleno dia, Plena Noite, Amanhecer, Anoitecer', 'sim', ORIGEM_OC + '; fase_dia; R3'],
    ['fato_acidente', 'condicao_tempo', 'categórico', 'Condição do tempo agrupada', None, None, 'Céu Claro, Nublado, Chuva, Nevoeiro/Neblina, Vento, Granizo, Neve', 'sim', ORIGEM_OC + '; condicao_metereologica; R3, R4'],
    ['fato_acidente', 'tipo_pista', 'categórico', 'Tipo de pista', None, None, 'Simples, Dupla, Múltipla', 'sim', ORIGEM_OC + '; tipo_pista; R3'],
    ['fato_acidente', 'tracado_via', 'texto', 'Traçado da via (pode ter vários, separados por ;)', None, None, None, 'sim', ORIGEM_OC + '; tracado_via; R3'],
    ['fato_acidente', 'uso_solo', 'categórico', 'Área urbana ou rural', None, None, 'Urbano, Rural', 'sim', ORIGEM_OC + '; uso_solo; R3'],
    ['fato_acidente', 'pessoas', 'numérico', 'Pessoas envolvidas', 1, 200, None, 'não', ORIGEM_OC + '; pessoas; R1, R8'],
    ['fato_acidente', 'mortos', 'numérico', 'Mortos no acidente', 0, 100, None, 'não', ORIGEM_OC + '; mortos; R1, R8'],
    ['fato_acidente', 'feridos_graves', 'numérico', 'Feridos graves', 0, 100, None, 'não', ORIGEM_OC + '; feridos_graves; R1, R8'],
    ['fato_acidente', 'veiculos', 'numérico', 'Veículos envolvidos', 0, 100, None, 'não', ORIGEM_OC + '; veiculos; R1'],
    ['fato_acidente', 'tem_moto', 'booleano (0/1)', 'Envolveu motocicleta, motoneta ou ciclomotor', 0, 1, None, 'não', ORIGEM_PE + '; tipo_veiculo; R12'],
    ['fato_acidente', 'tem_pedestre', 'booleano (0/1)', 'Envolveu pedestre', 0, 1, None, 'não', ORIGEM_PE + '; tipo_envolvido; R12'],
    ['fato_acidente', 'tem_pesado', 'booleano (0/1)', 'Envolveu caminhão, ônibus ou reboque', 0, 1, None, 'não', ORIGEM_PE + '; tipo_veiculo; R12'],
    ['fato_acidente', 'grave', 'booleano (0/1)', 'Ao menos 1 morto ou ferido grave', 0, 1, None, 'não', 'mortos e feridos_graves; R9'],
    ['fato_envolvido', 'id_acidente', 'numérico', 'Acidente em que a pessoa estava', None, None, None, 'não', ORIGEM_PE + '; id; R5'],
    ['fato_envolvido', 'tipo_veiculo', 'categórico', 'Veículo em que a pessoa estava', None, None, None, 'sim (pedestres)', ORIGEM_PE + '; tipo_veiculo; R3'],
    ['fato_envolvido', 'ano_fabricacao_veiculo', 'numérico', 'Ano de fabricação do veículo', 1950, 2025, None, 'sim', ORIGEM_PE + '; ano_fabricacao_veiculo; R7'],
    ['fato_envolvido', 'tipo_envolvido', 'categórico', 'Papel da pessoa', None, None, 'Condutor, Passageiro, Pedestre, Testemunha, Cavaleiro', 'sim', ORIGEM_PE + '; tipo_envolvido; R3'],
    ['fato_envolvido', 'estado_fisico', 'categórico', 'Estado da pessoa após o acidente', None, None, 'Ileso, Lesões Leves, Lesões Graves, Óbito', 'sim', ORIGEM_PE + '; estado_fisico; R3'],
    ['fato_envolvido', 'faixa_etaria', 'categórico', 'Faixa de idade (a idade exata é descartada)', None, None, '0-17, 18-29, 30-44, 45-59, 60+', 'sim', ORIGEM_PE + '; idade; R6, R7'],
    ['fato_envolvido', 'sexo', 'categórico', 'Sexo', None, None, 'Masculino, Feminino', 'sim', ORIGEM_PE + '; sexo; R3'],
    ['fato_envolvido', 'obito', 'booleano (0/1)', 'A pessoa morreu', 0, 1, None, 'não', ORIGEM_PE + '; mortos; R5'],
    ['dim_tempo', 'data', 'data', 'Dia do calendário', '2024-01-01', '2025-12-31', None, 'não', 'Gerado (calendário contínuo)'],
    ['dim_tempo', 'mes', 'numérico', 'Mês', 1, 12, None, 'não', 'Derivado de data'],
    ['dim_tempo', 'dia_semana', 'categórico', 'Dia da semana', None, None, ', '.join(nomes_dias), 'não', 'Derivado de data'],
    ['dim_tempo', 'fim_de_semana', 'booleano', 'Sábado ou domingo', None, None, 'True, False', 'não', 'Derivado de data; R11'],
    ['dim_tempo', 'feriado', 'booleano', 'Feriado ou ponto facultativo nacional', None, None, 'True, False', 'não', 'Datas fixas e Páscoa; R11'],
    ['dim_trecho', 'id_trecho', 'texto', 'BR, UF e km inicial do segmento', None, None, None, 'não', 'R10'],
    ['dim_trecho', 'br', 'numérico', 'Número da rodovia federal', 10, 499, None, 'não', ORIGEM_OC + '; br; R1'],
    ['dim_trecho', 'uf', 'categórico', 'Unidade da federação', None, None, 'AC, AL, AM, AP, BA, CE, DF, ES, GO, MA, MG, MS, MT, PA, PB, PE, PI, PR, RJ, RN, RO, RR, RS, SC, SE, SP, TO', 'não', ORIGEM_OC + '; uf; R3'],
    ['dim_trecho', 'regiao', 'categórico', 'Região do IBGE', None, None, 'Norte, Nordeste, Centro-Oeste, Sudeste, Sul', 'não', 'Derivado de uf; R11'],
    ['dim_trecho', 'km_inicial', 'numérico', 'Início do segmento de 10 km', 0, 1500, None, 'não', 'Derivado de km; R10'],
], columns=['tabela', 'atributo', 'tipo', 'descricao', 'minimo', 'maximo', 'categorias_validas', 'admite_nulo', 'linhagem'])

catalogo.to_csv(os.path.join(CATALOGO, 'catalogo_dados.csv'), index=False)
print(len(catalogo), 'atributos no catálogo')
catalogo.head(10)

## 7. Carga na nuvem e conferência

As tabelas da camada ouro são gravadas no Drive e **lidas de volta**. Todas as análises a
seguir usam as tabelas relidas, o que comprova que os dados estão persistidos na nuvem.

In [ ]:
tabelas = {'fato_acidente': fato_acidente, 'fato_envolvido': fato_envolvido,
           'dim_tempo': dim_tempo, 'dim_trecho': dim_trecho}
for nome, tabela in tabelas.items():
    tabela.to_csv(os.path.join(OURO, f'{nome}.csv'), index=False)

relidas = {}
for nome in tabelas:
    relidas[nome] = pd.read_csv(os.path.join(OURO, f'{nome}.csv'), low_memory=False)

conferencia_carga = pd.DataFrame({
    'tabela': list(tabelas.keys()),
    'linhas_gravadas': [len(t) for t in tabelas.values()],
    'linhas_relidas': [len(t) for t in relidas.values()],
})
conferencia_carga['ok'] = conferencia_carga['linhas_gravadas'] == conferencia_carga['linhas_relidas']
print('Soma de mortos gravada:', fato_acidente['mortos'].sum(), '| relida:', relidas['fato_acidente']['mortos'].sum())
conferencia_carga

In [ ]:
# Testes de domínio derivados do catálogo, feitos nas tabelas relidas
testes = []
for _, linha in catalogo.iterrows():
    coluna = relidas[linha['tabela']][linha['atributo']]
    if linha['admite_nulo'] == 'não':
        testes.append([linha['tabela'], linha['atributo'], 'sem nulos', coluna.notna().all()])
    if linha['tipo'] in ['numérico', 'booleano (0/1)'] and pd.notna(linha['minimo']):
        testes.append([linha['tabela'], linha['atributo'], f"entre {linha['minimo']} e {linha['maximo']}",
                       coluna.dropna().between(float(linha['minimo']), float(linha['maximo'])).all()])
    if linha['tipo'] == 'categórico' and pd.notna(linha['categorias_validas']):
        validas = [c.strip() for c in linha['categorias_validas'].split(',')]
        testes.append([linha['tabela'], linha['atributo'], 'categoria válida', coluna.dropna().astype(str).isin(validas).all()])
testes = pd.DataFrame(testes, columns=['tabela', 'atributo', 'teste', 'passou'])
testes.to_csv(os.path.join(CATALOGO, 'testes_dominio.csv'), index=False)
print(f"Testes aprovados: {testes['passou'].sum()} de {len(testes)}")
testes[~testes['passou']]

In [ ]:
# Tabelas usadas na análise (sempre a partir das tabelas relidas)
fato = relidas['fato_acidente'].copy()
fato['data'] = pd.to_datetime(fato['data'])
tempo = relidas['dim_tempo'].copy()
tempo['data'] = pd.to_datetime(tempo['data'])
trechos = relidas['dim_trecho']
envolvidos_ouro = relidas['fato_envolvido']

base = fato.merge(tempo, on='data', how='left').merge(trechos, on='id_trecho', how='left')
taxa_nacional = base['grave'].mean()
print(f'{len(base):,} acidentes, {base["grave"].sum():,} graves ({taxa_nacional * 100:.1f}%), {base["mortos"].sum():,} mortos')
base.groupby('ano')[['grave', 'mortos']].agg(['sum', 'mean']).round(3)

## 8. Análise: onde, quando, como e quem (P1 a P4)

In [ ]:
# P1 - ONDE: concentração dos acidentes graves nos trechos de 10 km
por_trecho = (base.dropna(subset=['id_trecho']).groupby('id_trecho')
              .agg(acidentes=('grave', 'size'), graves=('grave', 'sum'), mortos=('mortos', 'sum'))
              .sort_values('graves', ascending=False))
por_trecho['pct_acumulado_graves'] = por_trecho['graves'].cumsum() / por_trecho['graves'].sum() * 100
n_trechos = len(por_trecho)
top10 = int(np.ceil(n_trechos * 0.10))
pct_top10 = por_trecho['graves'].iloc[:top10].sum() / por_trecho['graves'].sum() * 100
trechos_metade = (por_trecho['pct_acumulado_graves'] < 50).sum() + 1
print(f'{n_trechos:,} trechos com acidentes. Os 10% mais críticos ({top10}) concentram {pct_top10:.1f}% dos acidentes graves.')
print(f'Metade dos acidentes graves acontece em {trechos_metade} trechos ({trechos_metade / n_trechos * 100:.1f}% do total).')
print(f'Trechos com pelo menos 1 acidente grave: {(por_trecho["graves"] > 0).sum():,}')

por_uf = base.groupby('uf').agg(acidentes=('grave', 'size'), graves=('grave', 'sum'), mortos=('mortos', 'sum'))
por_uf['taxa_grave_%'] = por_uf['graves'] / por_uf['acidentes'] * 100
por_uf['mortos_por_100_acidentes'] = por_uf['mortos'] / por_uf['acidentes'] * 100
por_uf.sort_values('graves', ascending=False).round(1).head(10)

In [ ]:
# Os trechos críticos se repetem de um ano para o outro? (validação fora do tempo)
graves_ano = (base.dropna(subset=['id_trecho']).pivot_table(index='id_trecho', columns='ano', values='grave', aggfunc='sum')
              .fillna(0))
graves_ano.columns = ['graves_2024', 'graves_2025']
ordem_2024 = graves_ano.sort_values('graves_2024', ascending=False)
simulador_trechos = []
for pct in [1, 5, 10, 20]:
    k = int(np.ceil(len(ordem_2024) * pct / 100))
    cobertura = ordem_2024['graves_2025'].iloc[:k].sum() / ordem_2024['graves_2025'].sum() * 100
    simulador_trechos.append([pct, k, round(cobertura, 1)])
simulador_trechos = pd.DataFrame(simulador_trechos, columns=['% dos trechos fiscalizados', 'trechos', '% dos graves de 2025 cobertos'])
correlacao = graves_ano['graves_2024'].corr(graves_ano['graves_2025'])

# Figura: curva de cobertura dos trechos
fig, ax = plt.subplots(figsize=(10, 3.2))
pcts = np.arange(0, 101, 1)
mesmo_periodo = [por_trecho['graves'].iloc[:int(np.ceil(len(por_trecho) * p / 100))].sum() / por_trecho['graves'].sum() * 100 for p in pcts]
fora_do_tempo = [ordem_2024['graves_2025'].iloc[:int(np.ceil(len(ordem_2024) * p / 100))].sum() / ordem_2024['graves_2025'].sum() * 100 for p in pcts]
ax.plot(pcts, mesmo_periodo, color=AZUL, lw=2, label='Concentração em 2024-2025 (P1)')
ax.plot(pcts, fora_do_tempo, color=LARANJA, lw=2, ls='--', label='Trechos escolhidos com 2024, graves de 2025 (simulador)')
ax.plot([0, 100], [0, 100], color=CINZA, lw=1, ls=':', label='Sem concentração')
for p in [5, 10]:
    ax.axvline(p, color=CINZA, lw=0.8)
ax.set_xlabel('% dos trechos de 10 km (do mais para o menos crítico)')
ax.set_ylabel('% dos acidentes graves')
ax.legend(frameon=False, fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(EVIDENCIAS, 'fig_trechos.png'), dpi=200, bbox_inches='tight')
plt.show()
pd.DataFrame({'pct_trechos': pcts, 'concentracao': mesmo_periodo, 'fora_do_tempo': fora_do_tempo}).to_csv(
    os.path.join(EVIDENCIAS, 'curva_trechos.csv'), index=False)
simulador_trechos.to_csv(os.path.join(EVIDENCIAS, 'simulador_trechos.csv'), index=False)
print(f'Correlação entre os acidentes graves de cada trecho em 2024 e em 2025: {correlacao:.2f}')
simulador_trechos

In [ ]:
# P2 - QUANDO: taxa de gravidade por fase do dia, faixa horária e dia da semana
taxa_fase = base.groupby('fase_dia')['grave'].agg(['size', 'mean']).rename(columns={'size': 'acidentes', 'mean': 'taxa_grave'})
taxa_dia = base.loc[base['fase_dia'] == 'Pleno dia', 'grave'].mean()
taxa_noite = base.loc[base['fase_dia'] == 'Plena Noite', 'grave'].mean()
razao_noite = taxa_noite / taxa_dia
print(f'Taxa de gravidade: dia {taxa_dia * 100:.1f}% | noite {taxa_noite * 100:.1f}% | razão {razao_noite:.2f}')

noturno = (base['hora'] >= 18) | (base['hora'] < 6)
print(f"Entre 18h e 6h: {noturno.mean() * 100:.1f}% dos acidentes e {noturno[base['grave'] == 1].mean() * 100:.1f}% dos graves")
taxa_hora = base.groupby('hora')['grave'].agg(['size', 'mean'])
taxa_semana = base.groupby('dia_semana')['grave'].agg(['size', 'mean']).reindex(nomes_dias)
taxa_feriado = base.groupby('feriado')['grave'].agg(['size', 'mean'])
media_dia_feriado = base[base['feriado']].groupby('data').size().mean()
media_dia_comum = base[~base['feriado']].groupby('data').size().mean()
print(f'Acidentes por dia: feriados {media_dia_feriado:.0f} | demais dias {media_dia_comum:.0f}')
print((taxa_fase['taxa_grave'] * 100).round(1))
(taxa_semana['mean'] * 100).round(1)

In [ ]:
# Figura: acidentes e taxa de gravidade por hora do dia
fig, eixos = plt.subplots(1, 2, figsize=(10, 3.2))
eixos[0].bar(taxa_hora.index, taxa_hora['size'] / len(ANOS) / 365, color=AZUL, width=0.8)
eixos[0].set_title('Acidentes por dia, por hora (média)', fontsize=10)
eixos[0].set_xlabel('Hora do dia')
eixos[1].plot(taxa_hora.index, taxa_hora['mean'] * 100, color=LARANJA, marker='o', ms=4, lw=2)
eixos[1].axhline(taxa_nacional * 100, color=CINZA, ls='--', lw=1, label=f'média ({taxa_nacional * 100:.1f}%)'.replace('.', ','))
eixos[1].set_title('Taxa de acidentes graves (%), por hora', fontsize=10)
eixos[1].set_xlabel('Hora do dia')
eixos[1].legend(frameon=False, fontsize=8)
for e in eixos:
    e.set_xticks(range(0, 24, 3))
plt.tight_layout()
plt.savefig(os.path.join(EVIDENCIAS, 'fig_hora.png'), dpi=200, bbox_inches='tight')
plt.show()
taxa_hora.to_csv(os.path.join(EVIDENCIAS, 'taxa_por_hora.csv'))

In [ ]:
# P3 - COMO: tipos de acidente e condições da via
por_tipo = (base.groupby('tipo_acidente')
            .agg(acidentes=('grave', 'size'), graves=('grave', 'sum'), mortos=('mortos', 'sum'))
            .sort_values('mortos', ascending=False))
por_tipo['% das mortes'] = por_tipo['mortos'] / por_tipo['mortos'].sum() * 100
por_tipo['% dos acidentes'] = por_tipo['acidentes'] / por_tipo['acidentes'].sum() * 100
por_tipo['taxa_grave_%'] = por_tipo['graves'] / por_tipo['acidentes'] * 100
pct_3_tipos = por_tipo['% das mortes'].iloc[:3].sum()
print(f"Os 3 tipos que mais matam ({', '.join(por_tipo.index[:3])}) respondem por {pct_3_tipos:.1f}% das mortes "
      f"e {por_tipo['% dos acidentes'].iloc[:3].sum():.1f}% dos acidentes.")
print((base.groupby('tipo_pista')['grave'].mean() * 100).round(1))
por_tipo.round(1).head(8)

In [ ]:
# P4 - QUEM: letalidade (óbitos / envolvidos) por tipo de envolvido e de veículo
env = envolvidos_ouro.dropna(subset=['estado_fisico'])
letalidade_media = env['obito'].mean()
env = env.assign(grupo=np.where(env['tipo_envolvido'] == 'Pedestre', 'Pedestre',
                       np.where(env['tipo_veiculo'].isin(motos), 'Motocicleta',
                       np.where(env['tipo_veiculo'] == 'Bicicleta', 'Bicicleta',
                       np.where(env['tipo_veiculo'].isin(['Automóvel', 'Caminhonete', 'Camioneta', 'Utilitário']), 'Carro e utilitário',
                       np.where(env['tipo_veiculo'].isin(pesados), 'Caminhão e ônibus', 'Outros'))))))
letalidade = env.groupby('grupo')['obito'].agg(envolvidos='size', obitos='sum', letalidade='mean')
letalidade['letalidade_%'] = letalidade['letalidade'] * 100
letalidade['razao_media'] = letalidade['letalidade'] / letalidade_media
letalidade['% dos óbitos'] = letalidade['obitos'] / letalidade['obitos'].sum() * 100
letalidade = letalidade.sort_values('letalidade_%', ascending=False)
print(f'Letalidade média: {letalidade_media * 100:.2f}% (envolvidos com estado físico informado)')
print('Letalidade por faixa etária (%):')
print((env.groupby('faixa_etaria')['obito'].mean() * 100).round(2))
letalidade.drop(columns='letalidade').round(2)

In [ ]:
# Figura: letalidade por grupo
fig, ax = plt.subplots(figsize=(10, 3))
grupos = letalidade.index
cores = [LARANJA if g in ['Pedestre', 'Motocicleta'] else AZUL for g in grupos]
barras = ax.barh(grupos[::-1], letalidade['letalidade_%'][::-1], color=cores[::-1], height=0.6)
for b, v in zip(barras, letalidade['letalidade_%'][::-1]):
    ax.text(v + 0.2, b.get_y() + b.get_height() / 2, f'{v:.1f}%'.replace('.', ','), va='center', fontsize=9)
ax.axvline(letalidade_media * 100, color=CINZA, ls='--', lw=1)
ax.set_xlabel('Letalidade: óbitos por 100 envolvidos')
plt.tight_layout()
plt.savefig(os.path.join(EVIDENCIAS, 'fig_letalidade.png'), dpi=200, bbox_inches='tight')
plt.show()
letalidade.to_csv(os.path.join(EVIDENCIAS, 'letalidade.csv'))
por_tipo.to_csv(os.path.join(EVIDENCIAS, 'tipos_acidente.csv'))

## 9. Machine learning: prever a gravidade no momento do chamado (P5)

**Decisão apoiada:** quando um acidente é comunicado, enviar ou não socorro prioritário.

**Variáveis de entrada:** só o que se sabe no momento do chamado: tipo de acidente, hora, fase do
dia, dia da semana, feriado, condição do tempo, tipo de pista, curva, área urbana ou rural, região,
número de veículos e de pessoas, e se há moto, pedestre ou veículo pesado.
A **causa** não entra, porque só é apurada depois, na investigação.

**Divisão no tempo:** treino com 2024 e teste com 2025 (dados que os modelos nunca viram).

**Modelos comparados:**
1. **Regra simples (referência):** a chance de ser grave é a taxa de 2024 do tipo de acidente.
2. **Regressão logística:** soma ponderada das variáveis, transformada em probabilidade.
3. **Árvore de decisão:** sequência de perguntas do tipo "houve pedestre?", "a pista é simples?".
4. **Random forest:** média de 300 árvores treinadas com amostras diferentes.
5. **Gradient boosting:** árvores pequenas em sequência, cada uma corrigindo a anterior.

**Métricas:** AUC (chance de o modelo dar nota maior a um acidente grave do que a um não grave;
0,5 = sorteio, 1 = perfeito) e **captura**: se a central só puder priorizar 30% dos chamados,
que porcentagem dos acidentes graves estaria entre eles.

In [ ]:
dados_ml = base.dropna(subset=['tipo_acidente', 'hora']).copy()
dados_ml['curva'] = dados_ml['tracado_via'].fillna('').str.contains('Curva').astype(int)
categoricas = ['tipo_acidente', 'fase_dia', 'dia_semana', 'condicao_tempo', 'tipo_pista', 'uso_solo', 'regiao']
numericas = ['hora', 'veiculos', 'pessoas', 'tem_moto', 'tem_pedestre', 'tem_pesado', 'curva', 'feriado']
X = pd.get_dummies(dados_ml[categoricas].fillna('não informado')).astype(int)
for coluna in numericas:
    X[coluna] = dados_ml[coluna].astype(int)
y = dados_ml['grave'].values
treino = (dados_ml['ano'] == 2024).values
teste = (dados_ml['ano'] == 2025).values
print(f'Treino: {treino.sum():,} acidentes | Teste: {teste.sum():,} | Variáveis: {X.shape[1]}')

In [ ]:
def captura(probabilidade, real, pct=30):
    """% dos acidentes graves que ficam entre os pct% de maior probabilidade."""
    ordem = np.argsort(-probabilidade, kind='stable')
    k = int(len(ordem) * pct / 100)
    return real[ordem[:k]].sum() / real.sum() * 100

# 1. Regra simples: taxa de gravidade de 2024 por tipo de acidente
taxa_tipo_2024 = dados_ml[treino].groupby('tipo_acidente')['grave'].mean()
regra = dados_ml.loc[teste, 'tipo_acidente'].map(taxa_tipo_2024).fillna(y[treino].mean()).values
probabilidades = {'Regra (tipo de acidente)': regra}

modelos = {
    'Regressão logística': LogisticRegression(max_iter=3000),
    'Árvore de decisão': DecisionTreeClassifier(max_depth=6, min_samples_leaf=50, random_state=42),
    'Random forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42, n_jobs=-1),
    'Gradient boosting': GradientBoostingClassifier(random_state=42),
}
for nome, modelo in modelos.items():
    modelo.fit(X[treino], y[treino])
    probabilidades[nome] = modelo.predict_proba(X[teste])[:, 1]

resultados = []
for nome, prob in probabilidades.items():
    resultados.append([nome, roc_auc_score(y[teste], prob), captura(prob, y[teste], 30)])
resultados = pd.DataFrame(resultados, columns=['modelo', 'AUC', 'captura_30%']).set_index('modelo')
MELHOR = resultados.drop('Regra (tipo de acidente)')['captura_30%'].idxmax()
ganho = resultados.loc[MELHOR, 'captura_30%'] - resultados.loc['Regra (tipo de acidente)', 'captura_30%']
print(f'Melhor modelo: {MELHOR} | ganho de captura sobre a regra: {ganho:.1f} pontos percentuais')
resultados.round(3)

In [ ]:
# Quais variáveis mais pesam no random forest?
importancia = pd.Series(modelos['Random forest'].feature_importances_, index=X.columns)
agrupada = importancia.groupby(lambda v: next((c for c in categoricas if v.startswith(c + '_')), v)).sum()
(agrupada.sort_values(ascending=False) * 100).round(1).head(10)

In [ ]:
# Simulador de priorização: quanto se captura priorizando k% dos chamados
simulador_ml = []
for pct in [10, 20, 30, 40, 50]:
    simulador_ml.append([pct,
                         round(captura(probabilidades['Regra (tipo de acidente)'], y[teste], pct), 1),
                         round(captura(probabilidades[MELHOR], y[teste], pct), 1)])
simulador_ml = pd.DataFrame(simulador_ml, columns=['% dos chamados priorizados', 'captura regra (%)', f'captura {MELHOR} (%)'])
simulador_ml['precisão do modelo (%)'] = [round(c * y[teste].mean() / p * 100, 1) for p, c in
                                          zip(simulador_ml['% dos chamados priorizados'], simulador_ml[f'captura {MELHOR} (%)'])]
simulador_ml

In [ ]:
# Figura: curva de captura
fig, ax = plt.subplots(figsize=(10, 3.2))
pcts = np.arange(0, 101, 2)
for nome, cor, estilo in [(MELHOR, AZUL, '-'), ('Regra (tipo de acidente)', LARANJA, '--')]:
    ax.plot(pcts, [captura(probabilidades[nome], y[teste], p) if p > 0 else 0 for p in pcts], color=cor, lw=2, ls=estilo, label=nome)
ax.plot([0, 100], [0, 100], color=CINZA, lw=1, ls=':', label='Sorteio')
ax.axvline(30, color=CINZA, lw=0.8)
ax.set_xlabel('% dos chamados priorizados (do maior para o menor risco previsto)')
ax.set_ylabel('% dos graves capturados')
ax.legend(frameon=False, fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(EVIDENCIAS, 'fig_captura.png'), dpi=200, bbox_inches='tight')
plt.show()
resultados.to_csv(os.path.join(EVIDENCIAS, 'resultados_ml.csv'))
simulador_ml.to_csv(os.path.join(EVIDENCIAS, 'simulador_ml.csv'), index=False)
pd.DataFrame({'pct_chamados': pcts,
              'modelo': [captura(probabilidades[MELHOR], y[teste], p) if p > 0 else 0 for p in pcts],
              'regra': [captura(probabilidades['Regra (tipo de acidente)'], y[teste], p) if p > 0 else 0 for p in pcts]}).to_csv(
    os.path.join(EVIDENCIAS, 'curva_captura.csv'), index=False)

## 10. Linha de tomada de decisão: gatilhos, painel e simuladores

| Gatilho | Limite | Decisão sugerida | Responsável e frequência |
|---|---|---|---|
| G1 | Trecho entre os 5% com mais acidentes graves no ano anterior | D1: fiscalização fixa e radar móvel no trecho | Planejamento operacional, anual com revisão trimestral |
| G2 | Taxa de gravidade do trecho ≥ 1,5 vez a nacional, com ≥ 10 acidentes | D2: vistoria de engenharia com o DNIT (sinalização, iluminação, duplicação) | Superintendência regional, semestral |
| G3 | ≥ 60% dos acidentes graves do trecho entre 18h e 6h (a média nacional é 46%), com ≥ 5 graves | D3: operação noturna no trecho | Delegacia, mensal |
| G4 | Chamado com risco previsto entre os 30% maiores | D4: despachar socorro prioritário | Central de atendimento, em tempo real |

In [ ]:
painel = (base.dropna(subset=['id_trecho']).groupby('id_trecho')
          .agg(acidentes=('grave', 'size'), graves=('grave', 'sum'), mortos=('mortos', 'sum'))
          .join(graves_ano))
noite = base[(base['grave'] == 1) & ((base['hora'] >= 18) | (base['hora'] < 6))].groupby('id_trecho').size()
painel['graves_noite'] = noite.reindex(painel.index).fillna(0)
painel['taxa_grave_%'] = painel['graves'] / painel['acidentes'] * 100
painel['% graves à noite'] = painel['graves_noite'] / painel['graves'].replace(0, np.nan) * 100

limite_g1 = ordem_2024['graves_2024'].iloc[int(np.ceil(len(ordem_2024) * 0.05)) - 1]
painel['G1'] = painel['graves_2024'] >= limite_g1
painel['G2'] = (painel['taxa_grave_%'] >= 1.5 * taxa_nacional * 100) & (painel['acidentes'] >= 10)
painel['G3'] = (painel['% graves à noite'] >= 60) & (painel['graves'] >= 5)
painel['gatilhos'] = (painel[['G1', 'G2', 'G3']].apply(lambda l: ', '.join(g for g in ['G1', 'G2', 'G3'] if l[g]), axis=1))
painel['prioridade'] = np.where(painel['G1'], 'Alta', np.where(painel['G2'] | painel['G3'], 'Média', 'Monitorar'))
print(f'Limite do G1: trecho com pelo menos {limite_g1:.0f} acidentes graves em 2024')
print(painel['prioridade'].value_counts())
cobertura_alta = painel.loc[painel['prioridade'] == 'Alta', 'graves_2025'].sum() / painel['graves_2025'].sum() * 100
cobertura_alta_media = painel.loc[painel['prioridade'] != 'Monitorar', 'graves_2025'].sum() / painel['graves_2025'].sum() * 100
print(f'Trechos de prioridade alta cobrem {cobertura_alta:.1f}% dos graves de 2025; alta + média: {cobertura_alta_media:.1f}%')
print(painel.groupby('prioridade')[['G1', 'G2', 'G3']].sum())

painel_alta = (painel[painel['prioridade'] == 'Alta'].sort_values('graves', ascending=False)
               [['acidentes', 'graves', 'mortos', 'taxa_grave_%', '% graves à noite', 'gatilhos']])
painel.to_csv(os.path.join(EVIDENCIAS, 'painel_trechos.csv'))
painel_alta.round(1).head(12)

## 11. Verificação da hipótese

In [ ]:
verificacao = pd.DataFrame([
    ['P1', '10% dos trechos concentram ≥ 40% dos graves', f'{pct_top10:.1f}%', pct_top10 >= 40],
    ['P2', 'Taxa à noite ≥ 1,2 × a de dia', f'{razao_noite:.2f} ×', razao_noite >= 1.2],
    ['P3', '3 tipos respondem por ≥ 50% das mortes', f'{pct_3_tipos:.1f}%', pct_3_tipos >= 50],
    ['P4', 'Pedestres e motociclistas com letalidade ≥ 2 × a média',
     f"{letalidade.loc['Pedestre', 'razao_media']:.1f} × e {letalidade.loc['Motocicleta', 'razao_media']:.1f} ×",
     (letalidade.loc['Pedestre', 'razao_media'] >= 2) and (letalidade.loc['Motocicleta', 'razao_media'] >= 2)],
    ['P5', 'Melhor modelo captura ≥ 5 p.p. a mais que a regra (30% priorizados)', f'+{ganho:.1f} p.p. ({MELHOR})', ganho >= 5],
], columns=['pergunta', 'critério', 'resultado', 'atendido'])
verificacao.to_csv(os.path.join(EVIDENCIAS, 'verificacao_hipotese.csv'), index=False)
print('Hipótese confirmada:', bool(verificacao['atendido'].all()))
verificacao

## 12. Autoavaliação

A autoavaliação completa está no `README.md` do repositório e na conclusão do short paper:
quais perguntas foram respondidas, as limitações dos dados, o que seria feito de outra forma e
o que falta para o MVP virar uma solução de uso contínuo.